# AutoETF Condition Research MVP

第一版 MVP：检验 ETF 在单日满足“量比 > 1、换手率 < 5、今日上涨”后，下一交易日上涨概率是否高于全样本基准概率。

注意：本 Notebook 仅用于历史统计研究，不构成任何投资建议。

In [ ]:
from src.condition_parser import parse_condition_research
from src.data_probe_bigquant import run_data_probe
from src.data_loader_bigquant import load_condition_research_data_bigquant
from src.condition_analysis import run_conditional_probability_test, diagnose_condition_result
from src.condition_report import generate_condition_report, save_condition_report
from src.config import DEFAULT_CONFIG

config = DEFAULT_CONFIG

In [ ]:
def main():
    user_idea = "当单日量比 > 1、换手率 < 5、今日上涨时，明天这个 ETF 上涨几率大吗？"

    hypothesis = parse_condition_research(user_idea)  # [AI-CORE]

    # 第一次在 BigQuant 环境中运行时，必须先探测字段。
    probe_results = run_data_probe()

    df = load_condition_research_data_bigquant(
        start_date=config.start_date,
        end_date=config.end_date,
        table_name=config.fund_table,
        volume_col="volume",
        turnover_col="turnover",
    )

    result = run_conditional_probability_test(
        df=df,
        conditions=hypothesis["conditions"],
        target=hypothesis["target"],
        min_event_count=200,
    )
    diagnosis = diagnose_condition_result(result)  # [AI-CORE]
    report = generate_condition_report(hypothesis, result, diagnosis)  # [AI-CORE]
    save_condition_report(report)

    return {
        "hypothesis": hypothesis,
        "probe_tables": list(probe_results.keys()),
        "data_shape": df.shape,
        "event_count": result["event_count"],
        "event_up_probability": result["event_up_probability"],
        "baseline_up_probability": result["baseline_up_probability"],
        "probability_lift": result["probability_lift"],
        "diagnosis": diagnosis,
        "report": report,
    }

result = main()
result